# 2. Advection and resolution

A translating density profile is a useful way to see numerical diffusion. We run the same periodic advection problem at several resolutions and compare the final profiles.

Each run uses a temporary directory, so changing the resolution does not modify the example files.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO = Path('/content/RadHydropy')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/tkc004/RadHydropy.git', str(REPO)], check=True)
PINNED = ['numpy==2.5.3', 'scipy==1.18.1', 'matplotlib==3.10.9', 'h5py==3.15.1', 'PyYAML==6.0.3', 'unyt==3.0.4']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINNED, '-e', '.[docs]'], cwd=REPO, check=True)
sys.path.insert(0, str(REPO))
%matplotlib inline
import matplotlib.pyplot as plt
import yaml
from example import example_utils
from example.Advection1D import advection1d
plt.rcParams['figure.figsize'] = (9, 5)

In [ ]:
# Make one isolated configuration for each resolution.
base_yaml = REPO / 'example' / 'Advection1D' / 'advection1d.yaml'
base = yaml.safe_load(base_yaml.read_text())
results = {}

for resolution in (50, 100, 200):
    run_dir = Path('/content') / f'radhydropy-advection-{resolution}'
    if run_dir.exists():
        shutil.rmtree(run_dir)
    run_dir.mkdir()
    config = yaml.safe_load(yaml.safe_dump(base))
    config['par']['mesh']['grid_cells'] = resolution
    config['initial_condition']['grid_cells'] = resolution
    config['par']['output']['directory'] = '.'
    config['par']['simulation']['initial_condition_filename'] = 'InitialCondition.hdf5'
    config['example']['output_indices'] = [0, 5]
    config_file = run_dir / 'advection.yaml'
    config_file.write_text(yaml.safe_dump(config))
    advection1d.main(config_file)
    loaded = example_utils.load_nested_example_config(config_file)
    fields = example_utils.snapshot_physical_fields(run_dir / 'Output_005.hdf5', loaded)
    results[resolution] = fields

print('Completed resolutions:', sorted(results))

In [ ]:
for resolution, fields in results.items():
    x = fields['radius_proper_code']
    rho = fields['rho_proper_code']
    plt.plot(x, rho, label=f'{resolution} cells')
plt.xlabel('position [code length]')
plt.ylabel('density [code units]')
plt.title('Numerical diffusion in periodic advection')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## Interpretation

The profile should translate around the periodic domain. At lower resolution, sharp features spread over more cells: this is numerical diffusion. Increasing the resolution reduces the width of the smeared feature, at the cost of more cells and timesteps.

In [ ]:
# Optional: save all resolution runs to Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# save_dir = Path('/content/drive/MyDrive/RadHydropy/02_advection_and_resolution')
# save_dir.mkdir(parents=True, exist_ok=True)
# for resolution in results:
#     destination = save_dir / f'{resolution}_cells'
#     shutil.copytree(Path('/content') / f'radhydropy-advection-{resolution}', destination, dirs_exist_ok=True)
# print('Saved to:', save_dir)